# Home Credit Default Risk — 01. Load and audit data

Notebook này chạy trực tiếp trên Kaggle. Code tái sử dụng được clone từ GitHub; dữ liệu được đọc từ Kaggle Input và không được ghi vào Git.

## Chuẩn bị trên Kaggle

1. Tạo notebook mới và bật **Internet** để cell `git clone` hoạt động.
2. Chọn **Add Input** → tìm competition `Home Credit Default Risk` → chấp nhận rules nếu Kaggle yêu cầu.
3. Chạy notebook từ trên xuống. Pandas đã có sẵn trong Kaggle image, không cần `pip install`.

In [ ]:
# 1. Clone hoặc cập nhật source code từ GitHub
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ManhTanTran/Qaci-datascience.git"
REPO_DIR = Path("/kaggle/working/Qaci-datascience")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Source code: {REPO_DIR}")

In [ ]:
# 2. Import thư viện và loader của project
import pandas as pd

from credit_scoring.data import audit_home_credit_data, load_home_credit_data

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 160)

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")

In [ ]:
# 3. Xác định thư mục dữ liệu do Kaggle mount ở chế độ read-only
DATA_DIR = Path("/kaggle/input/home-credit-default-risk")
EXPECTED_FILES = {
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
    "installments_payments.csv",
}

if not DATA_DIR.is_dir():
    available_inputs = sorted(str(path) for path in Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(
        f"Không tìm thấy {DATA_DIR}. Hãy Add Input của competition. "
        f"Input hiện có: {available_inputs}"
    )

available_files = {path.name for path in DATA_DIR.glob("*.csv")}
missing_files = sorted(EXPECTED_FILES - available_files)
if missing_files:
    raise FileNotFoundError(f"Competition input thiếu file: {missing_files}")

print(f"Data directory: {DATA_DIR}")
print(f"Đã tìm thấy {len(available_files)} CSV files")

In [ ]:
# 4. Cấu hình lần chạy
# Đặt NROWS = 1_000 để smoke test nhanh; dùng None cho dữ liệu đầy đủ.
NROWS = None

# Bước đầu chỉ đọc application train/test để kiểm soát RAM và xây baseline.
# Sau này đổi thành TABLES = "all" để đọc toàn bộ bảng quan hệ.
TABLES = None

data = load_home_credit_data(
    data_dir=DATA_DIR,
    tables=TABLES,
    nrows=NROWS,
    reduce_memory=True,
    validate=True,
)

application_train = data["application_train"]
application_test = data["application_test"]

In [ ]:
# 5. Structural audit — không in dữ liệu từng khách hàng
audit = audit_home_credit_data(data)
display(audit)

target_distribution = (
    application_train["TARGET"]
    .value_counts(dropna=False)
    .rename_axis("TARGET")
    .to_frame("count")
)
target_distribution["ratio"] = target_distribution["count"] / len(application_train)
display(target_distribution)

print("Dtype counts:")
display(application_train.dtypes.value_counts().to_frame("column_count"))

In [ ]:
# 6. Assertions bắt buộc trước EDA/modeling
assert "TARGET" in application_train.columns
assert "TARGET" not in application_test.columns
assert application_train["SK_ID_CURR"].is_unique
assert application_test["SK_ID_CURR"].is_unique
assert set(application_train["TARGET"].dropna().unique()).issubset({0, 1})

overlap = set(application_train["SK_ID_CURR"]).intersection(application_test["SK_ID_CURR"])
assert not overlap, f"Train/test có {len(overlap)} SK_ID_CURR trùng nhau"

print("Load và kiểm tra cấu trúc hoàn tất. Sẵn sàng cho EDA và baseline.")

## Bước tiếp theo

Notebook tiếp theo sẽ xử lý sentinel `DAYS_EMPLOYED`, tạo application-level ratios, thiết lập 5-fold OOF và huấn luyện LightGBM baseline. Không ghi metric vào experiment log cho đến khi notebook thực sự được chạy và artifact được lưu.